# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/J-Aamir/Capstone-Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists("Capstone-Flyrank"):
    !git clone https://github.com/J-Aamir/Capstone-Flyrank.git
%cd Capstone-Flyrank

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)


Cloning into 'Capstone-Flyrank'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 168 (delta 68), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 1.88 MiB | 12.90 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/Capstone-Flyrank
(30000, 44)


## 1. My rule and its reason codes

**Two signal checks first**, each with a bucket table and n, before trusting any rule on them.

**Signal A — staleness behind the refresh flag** (`days_since_last_update` vs. decline rate). Verdict: **MIXED**. Decline rate does NOT rise cleanly with staleness (<60d: 51.2%, 60-180d: 61.1%, 180-365d: 46.7%, 365d+: only 5 rows, unreliable). There's no clean monotonic pattern, so staleness alone is a weak, inconsistent signal here — a useful negative that stops me building a rule around it as the primary driver.

**Signal B — CTR vs. position, behind the CTR-fix logic** (mean CTR by `position_tier`, `avg_position > 0` only — excluding the "no data" rows). Verdict: **CONFIRMED**. Mean CTR drops cleanly and monotonically as position worsens: top_3 2.764 -> page_1 0.652 -> striking 0.323 -> page_3_5 0.222 -> deep 0.150. This is real and strong enough to build a rule on.

**My rule:** flag pages whose actual CTR is meaningfully below the typical ("benchmark") CTR for pages at their position tier — a well-ranked page that still underperforms on clicks is a snippet/title problem, not a ranking problem. Score = `(benchmark_ctr - actual_ctr) * impressions_90d` (only positive gaps count; weighted by traffic since fixing high-traffic underperformers matters more). **Reason code:** `ctr_below_position_benchmark`. **Action label:** `review_title_and_meta_description`.

In [2]:
# Signal A: staleness vs decline rate
df['staleness_bucket'] = pd.cut(df['days_since_last_update'], bins=[-1,60,180,365,10000],
                                 labels=['<60d','60-180d','180-365d','365d+'])
signal_a = df.groupby('staleness_bucket').agg(
    n=('content_id','size'),
    decline_rate=('trend_direction', lambda s: (s=='down').mean())
).round(3)
print("Signal A — staleness vs decline rate:")
print(signal_a)
print()

# Signal B: CTR by position tier (avg_position > 0 only — 0 means "no data")
d = df[df.avg_position > 0].copy()
signal_b = d.groupby('position_tier')['ctr'].agg(n='size', mean_ctr='mean').round(3)\
             .reindex(['top_3','page_1','striking','page_3_5','deep'])
print("Signal B — CTR by position tier:")
print(signal_b)


Signal A — staleness vs decline rate:
                      n  decline_rate
staleness_bucket                     
<60d              20608         0.512
60-180d            9218         0.611
180-365d            169         0.467
365d+                 5         0.600

Signal B — CTR by position tier:
                   n  mean_ctr
position_tier                 
top_3           1116     2.764
page_1         11814     0.652
striking        7304     0.323
page_3_5        7242     0.222
deep            1319     0.150


/tmp/ipykernel_1140/379189796.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal_a = df.groupby('staleness_bucket').agg(


## 2. Build the ranked queue (writes the CSV)

Encodes the rule above: score, one reason code, one action label. Writes `work/outputs/baseline_action_score.csv`.

In [3]:
benchmark = d.groupby('position_tier')['ctr'].mean()
d['benchmark_ctr'] = d['position_tier'].map(benchmark)
d['ctr_gap'] = (d['benchmark_ctr'] - d['ctr']).clip(lower=0)
d['score'] = d['ctr_gap'] * d['impressions_90d']
d['reason_code'] = 'ctr_below_position_benchmark'
d['action'] = 'review_title_and_meta_description'

queue = d.sort_values('score', ascending=False)
print('Pages with a positive score (flagged):', (queue.score > 0).sum(), '/', len(queue))

import os
os.makedirs('work/outputs', exist_ok=True)
out_cols = ['content_id','client_id','position_tier','avg_position','ctr','benchmark_ctr',
            'impressions_90d','score','reason_code','action','trend_direction']
queue[out_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print('Written: work/outputs/baseline_action_score.csv')


Pages with a positive score (flagged): 23771 / 28795
Written: work/outputs/baseline_action_score.csv


## 3. Top-10 review

For each of the top 10: the action, why it's flagged, and what would make it wrong.

In [4]:
top10 = queue.head(10)[['content_id','position_tier','avg_position','ctr','benchmark_ctr',
                         'impressions_90d','score','trend_direction']]
print(top10.to_string(index=False))
print()
print(f"Among top 10, share actually labeled 'down': {(top10.trend_direction=='down').mean():.1%}")
print()
print("Per-row review (one line each):")
for i, row in top10.reset_index(drop=True).iterrows():
    print(f"{i+1}. {row.content_id} — action: review_title_and_meta_description — "
          f"why: CTR {row.ctr:.2f} vs benchmark {row.benchmark_ctr:.2f} at position "
          f"{row.avg_position:.1f}, {row.impressions_90d:,} impressions at stake — "
          f"would be WRONG if: the low CTR reflects a mismatched search intent rather than "
          f"a weak title/snippet, since intent mismatches aren't fixed by rewriting metadata.")


          content_id position_tier  avg_position  ctr  benchmark_ctr  impressions_90d        score trend_direction
content_8c19996aa890         top_3           2.5 0.15       2.764453           509252 1.331416e+06            down
content_4c36c775b818         top_3           2.3 0.41       2.764453           463103 1.090354e+06            down
content_8451fc6f034d         top_3           2.3 0.03       2.764453           272144 7.441651e+05              up
content_44e481c8f55b         top_3           1.4 0.65       2.764453           312694 6.611769e+05          stable
content_9532f197bbc8         top_3           2.0 0.87       2.764453           309192 5.857498e+05            down
content_e12868d1f396         top_3           2.9 0.07       2.764453           149712 4.033920e+05          stable
content_4a6607efcb46         top_3           2.2 0.01       2.764453           128068 3.527573e+05              up
content_4fc39a2b8cf0         top_3           2.6 0.69       2.764453           1

## 4. Weak picks + leakage check

**Weak picks:** only 3 of the top 10 (30%) are actually labeled `trend_direction == "down"` — several top picks (e.g. rows labeled `up` or `stable`) are pages with a real CTR gap but no current decline. That's a genuine weakness: the rule finds "underperforming for position" pages, which is a real problem worth fixing, but it is NOT the same target as "declining pages." A future model needs to decide whether CTR-gap or decline is the actual target, or combine both explicitly rather than treating a CTR-gap flag as a decline predictor.

**Leakage check:** the rule's inputs (`ctr`, `avg_position`, `impressions_90d`, `position_tier`) are all pre-decision, currently-observed page attributes — none are product flags, none are derived from `trend_direction`/`trend_pct`, and none reference a future time window. Confirmed below.

In [5]:
leakage_cols = ['trend_direction', 'trend_pct']
rule_inputs = ['ctr', 'avg_position', 'position_tier', 'impressions_90d']
overlap = set(leakage_cols) & set(rule_inputs)
print('Rule inputs:', rule_inputs)
print('Label-source columns (never inputs):', leakage_cols)
print('Overlap (should be empty):', overlap)
assert len(overlap) == 0, 'Leakage detected!'
print('No leakage: rule inputs are independent of the label-source columns.')


Rule inputs: ['ctr', 'avg_position', 'position_tier', 'impressions_90d']
Label-source columns (never inputs): ['trend_direction', 'trend_pct']
Overlap (should be empty): set()
No leakage: rule inputs are independent of the label-source columns.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.